In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────
# Installs are quiet (-q) so the output stays readable on a projector.
!pip install -q google-genai

import os, json, time                    # standard library
from google import genai                 # the SDK
from google.genai import types           # config and content types


# ── Your API key ──────────────────────────────────────────────────────
# The key lives in Colab Secrets, never in the notebook. If this cell
# fails, that is almost always why.
try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
    if not API_KEY:
        raise ValueError("empty")
except Exception:
    raise SystemExit(
        "\n" + "=" * 68 +
        "\nNo API key found.\n"
        "\n  1. Click the KEY icon in the left sidebar of Colab."
        "\n  2. Click 'Add new secret'."
        "\n  3. Name it exactly:  GEMINI_API_KEY"
        "\n  4. Paste your key from aistudio.google.com"
        "\n  5. Turn ON 'Notebook access' for this notebook."
        "\n  6. Run this cell again."
        "\n" + "=" * 68
    )

client = genai.Client(api_key=API_KEY)

MODEL = "gemini-2.5-flash-lite"          # fast and cheap; what we use all week
EMBED_MODEL = "gemini-embedding-001"    # free tier, which is what makes Day 2 possible

print("Ready. Model:", MODEL)

In [ ]:
# Tool 1 — a real function. Nothing AI about it.
def calculate(expression: str) -> str:
    """Evaluate a simple arithmetic expression and return the result."""
    # Only arithmetic characters allowed. The model can produce ANY string,
    # so validating the argument is your job, not the model's.
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        return json.dumps({"error": "Only arithmetic is allowed: digits and + - * / ( )"})
    try:
        return json.dumps({"result": eval(expression)})    # noqa: S307 - validated above
    except Exception as e:
        return json.dumps({"error": f"Could not evaluate: {e}"})


print(calculate("(420 + 75) * 3"))
print(calculate("import os"))          # rejected, as it should be

In [ ]:
# Tool 2 — deliberately mocked, with a fixed dataset.
WEATHER = {
    "riyadh": {"c": 41, "sky": "clear"},
    "jeddah": {"c": 34, "sky": "humid"},
    "abha":   {"c": 22, "sky": "cloudy"},
}


def get_weather(city: str) -> str:
    """Return today's weather for a Saudi city."""
    data = WEATHER.get(city.strip().lower())
    if not data:
        # A GOOD error: says what was wrong AND what a valid input looks like,
        # so the model can recover instead of hallucinating.
        return json.dumps({
            "error": f"No weather data for '{city}'.",
            "known_cities": sorted(WEATHER.keys()),
        })
    return json.dumps({"city": city, "celsius": data["c"], "sky": data["sky"]})


print(get_weather("Riyadh"))
print(get_weather("Paris"))

### Why a mock is fine here

`get_weather` returns made-up data and that is deliberate. **The loop is the
lesson, not the API.** Wiring a real weather service would add an account, a
key and a failure mode, and would teach you nothing about agents.

In your own project, replace the body with a real call. Everything around it —
the declaration, the loop, the guardrails — stays exactly the same.

Notice what `get_weather` returns when it fails: a message the *model* can
read, listing valid inputs. Error messages are prompts. Write them for the
model, not for your log file.

In [ ]:
# Tool declarations — the menu the model reads. Walked argument by argument.
calc_decl = types.FunctionDeclaration(
    name="calculate",                     # must match your function name
    description=(                         # ← THE most important field
        "Evaluate an arithmetic expression and return the numeric result. "
        "Use this for any calculation instead of doing arithmetic yourself."
    ),
    parameters={
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "Arithmetic only, e.g. '(420 + 75) * 3'",
            },
        },
        "required": ["expression"],
    },
)

weather_decl = types.FunctionDeclaration(
    name="get_weather",
    description=(
        "Get today's weather for a Saudi city. Returns temperature in "
        "Celsius and sky conditions. Only covers Riyadh, Jeddah and Abha."
    ),
    parameters={
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "City name, e.g. 'Riyadh'"},
        },
        "required": ["city"],
    },
)

TOOLS = {"calculate": calculate, "get_weather": get_weather}
tool_config = types.Tool(function_declarations=[calc_decl, weather_decl])
CFG = types.GenerateContentConfig(tools=[tool_config], temperature=0.0)

print("Declared:", list(TOOLS))

In [ ]:
# THE ROUND TRIP, STEP 1 OF 4 — ask. The model does NOT answer the question.
history = [types.Content(role="user", parts=[types.Part.from_text(
    text="What is the temperature in Jeddah, and what is that in Fahrenheit?")])]

r1 = client.models.generate_content(model=MODEL, contents=history, config=CFG)

print("Any text answer?", repr(r1.text))
print("It did not answer. It made a request. Look at the next cell.")

In [ ]:
# STEP 2 OF 4 — inspect what came back. This is a REQUEST, not an answer.
part = r1.candidates[0].content.parts[0]
call = part.function_call

print("tool requested :", call.name)
print("arguments      :", dict(call.args))
print()
print("Nothing has executed. No function has run. This is a message.")

In [ ]:
# STEP 3 OF 4 — YOUR code executes it. Delete this cell and nothing happens.
fn = TOOLS[call.name]                      # allow-list lookup, not eval
result = fn(**dict(call.args))

print("executed :", call.name)
print("returned :", result)
print()
print("This is also where you would check whether this user is allowed "
      "to call this tool with these arguments.")

In [ ]:
# STEP 4 OF 4 — hand the result back and ask again.
history.append(r1.candidates[0].content)          # what the model said
history.append(types.Content(role="user", parts=[  # what your code found
    types.Part.from_function_response(
        name=call.name, response={"result": result})]))

r2 = client.models.generate_content(model=MODEL, contents=history, config=CFG)

# It may want a second tool (the Fahrenheit conversion). That is the loop.
part2 = r2.candidates[0].content.parts[0]
if getattr(part2, "function_call", None):
    print("It wants another tool:", part2.function_call.name,
          dict(part2.function_call.args))
else:
    print(r2.text)

### Say it again: the model never ran your function

Four cells, and the only thing that executed any code was **cell 8**, which
you wrote and control.

The model produced a message that said *"I would like `get_weather` with
`city='Jeddah'`"*. Your application read that message and chose to honour it.

That choice point is the only place where permissions can be enforced. It is
also, on Thursday, the difference between a prompt injection being a nuisance
and being a breach — because whatever the attacker can make the model *ask*
for, your code decides whether to actually *do*.

In [ ]:
# The agent loop. GIVEN COMPLETE — read it, do not retype it.
def run_agent(goal, max_steps=5, verbose=True):
    """Loop until the model answers, or until the step cap fires."""
    history = [types.Content(role="user", parts=[types.Part.from_text(text=goal)])]

    for step in range(max_steps):
        r = client.models.generate_content(model=MODEL, contents=history, config=CFG)
        part = r.candidates[0].content.parts[0]

        # No tool requested → it is finished.
        if not getattr(part, "function_call", None):
            if verbose:
                print(f"  step {step}: final answer")
            return r.text

        call = part.function_call
        if verbose:
            print(f"  step {step}: {call.name}({dict(call.args)})")

        fn = TOOLS.get(call.name)                       # allow-list
        result = fn(**dict(call.args)) if fn else json.dumps(
            {"error": f"Unknown tool '{call.name}'.", "available": list(TOOLS)})

        if verbose:
            print(f"           → {result}")

        history.append(r.candidates[0].content)
        history.append(types.Content(role="user", parts=[
            types.Part.from_function_response(name=call.name,
                                              response={"result": result})]))

    return "Stopped: step limit reached without a final answer."

In [ ]:
# A two-step task, with the trace printed.
print(run_agent(
    "What is the temperature in Abha, and what is that in Fahrenheit? "
    "Use the calculator for the conversion.",
    max_steps=5,
))

In [ ]:
# TODO ─ Write a third tool and register it.
#
# The pattern is written three times above. Follow it:
#   1. write the function          2. write the declaration
#   3. add both to TOOLS and the tool config

def convert_currency(amount: float, to_currency: str) -> str:
    """Convert an amount in SAR to another currency."""
    RATES = {"usd": 0.267, "eur": 0.245, "gbp": 0.211}
    # ← TODO (2 lines): look up the rate, and return JSON with the converted
    #   amount. Return a helpful error (listing valid currencies) if unknown.
    return json.dumps({"error": "not implemented yet"})


currency_decl = types.FunctionDeclaration(
    name="convert_currency",
    description="TODO: write a description the model can act on",   # ← TODO
    parameters={
        "type": "object",
        "properties": {
            "amount": {"type": "number", "description": "Amount in SAR"},
            "to_currency": {"type": "string", "description": "usd, eur or gbp"},
        },
        "required": ["amount", "to_currency"],
    },
)

TOOLS["convert_currency"] = convert_currency
tool_config = types.Tool(function_declarations=[calc_decl, weather_decl, currency_decl])
CFG = types.GenerateContentConfig(tools=[tool_config], temperature=0.0)

print(run_agent("How much is 1485 SAR in US dollars?", max_steps=4))

In [ ]:
# Take the cap off and give it a goal it cannot satisfy.
# INTERRUPT THIS CELL YOURSELF (the stop button) after a few steps.
#
# max_steps=40 is not "no cap" — it is a cap high enough to be expensive,
# which is the point. Never run an actually-unbounded loop on a paid key.

print(run_agent(
    "What is the weather in Paris, France? Keep trying until you find it.",
    max_steps=40,
))

### What that would have cost

Count the steps you let it run. Now notice something: **the conversation is
re-sent on every step**, so step twenty is far more expensive than step one.
The cost curve is not flat, it accelerates.

Ten steps at roughly 4,000 input tokens each is 40,000 tokens for a question
that had no answer. On a free tier you hit a rate limit and stop — that is
luck, not design. On a paid key, running overnight, this is the invoice that
ends up in a post-mortem.

Every agent gets a step cap. Every one. Also worth adding: a stop condition
for repeated identical calls, and a wall-clock timeout.

In [ ]:
# Cap restored, plus an output validator.
ANSWER_SCHEMA = {
    "type": "object",
    "properties": {
        "answer": {"type": "string"},
        "confident": {"type": "boolean"},
    },
    "required": ["answer", "confident"],
}


def validated_agent(goal, max_steps=5):
    raw = run_agent(goal, max_steps=max_steps, verbose=False)

    # Force the final answer through a schema — Day 1's lesson, used as a
    # control. A shape you can check is a shape you can act on.
    r = client.models.generate_content(
        model=MODEL,
        contents=f"Rewrite this as structured output:\n{raw}",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=ANSWER_SCHEMA,
            temperature=0.0,
        ),
    )
    data = json.loads(r.text)

    if not data["confident"]:
        return "I could not answer that reliably."     # fail honestly
    return data["answer"]


print(validated_agent("What is the weather in Riyadh?"))
print(validated_agent("What is the weather in Paris?"))

## The ceiling — agentic RAG

Everything you built yesterday becomes **one tool** that today's agent can
choose to call.

That is the payoff of the whole week. Plain retrieval always searches, always
once, even when the question needs no documents or needs three different
searches. An agent decides *whether* to search, *what* to search for, and
whether the first result was good enough.

The next cell assumes you have Notebook 2 open in another tab. Copy your
`hybrid_search` function (and the `records` it closes over) into this
notebook, or re-run Notebook 2's Part A cells here. The `search_documents`
wrapper below is what turns it into a tool.

In [ ]:
# Your Day 2 retriever, wrapped as a tool.
#
# Paste your hybrid_search from Notebook 2 above this cell first. This stub
# keeps the notebook runnable if you have not yet.

def search_documents(query: str) -> str:
    """Search the policy documents. Returns passages with their sources."""
    try:
        hits = hybrid_search(query, k=3)          # from Notebook 2
    except NameError:
        return json.dumps({"error": "Retriever not loaded. "
                                    "Paste hybrid_search from Notebook 2 first."})
    # Truncate: every passage is re-sent on every later step of the loop.
    return json.dumps([{"text": h["text"][:600],
                        "source": h["source"], "page": h["page"]} for h in hits])


search_decl = types.FunctionDeclaration(
    name="search_documents",
    description=("Search the organisation's policy documents and return the "
                 "most relevant passages with their source and page. Use for "
                 "any question about internal policy, procedure or entitlement."),
    parameters={
        "type": "object",
        "properties": {
            "query": {"type": "string",
                      "description": "Search terms, not the raw question"},
        },
        "required": ["query"],
    },
)

TOOLS["search_documents"] = search_documents
tool_config = types.Tool(function_declarations=[calc_decl, search_decl])
CFG = types.GenerateContentConfig(tools=[tool_config], temperature=0.0)

# A question that needs BOTH retrieval and arithmetic. Watch the trace.
print(run_agent(
    "How many annual leave days does grade 11 get, and how many would be "
    "left after taking 12 days? Cite the source.",
    max_steps=6,
))

## Reflection

Fill these in before you close the notebook. This is what I check when I come round.

**In your own words: what executed your function, and what did the model actually do?**

> _your answer here_

**How many steps did the agentic RAG question take, and was that the minimum?**

> _your answer here_

**Name one task at your work where an agent is the right answer — and one where a fixed sequence would be better.**

> _your answer here_

## If this breaks

The three most likely failures, and what to do about each.

| Symptom | Cause | Fix |
|---|---|---|
| Model asks for the same tool over and over | The tool result was never appended to `history` | Both appends in the loop are required: the model's message AND the function response |
| `TypeError` when calling the tool | Schema type does not match the Python signature | `"type": "number"` for floats, `"integer"` for ints, `"string"` for text |
| The loop never ends | The goal cannot be satisfied with the tools available | That is cell 14, on purpose. Keep `max_steps` and add a repeated-call stop condition |